# ST-Reason · 03 Events, state confirmation and captions

For every video already processed by notebook 02, this notebook:

1. **Splits the video into 3 s windows that do not overlap.** The pilot measured 10.4 s per window with overlapping windows and JSON output, which projected 37 GPU hours. Non-overlapping windows, a short line format and batches of windows cut that by several times.
2. **Asks Qwen3-VL-4B, once per window,** using 4 frames with the tracks outlined and their IDs written on them, for:
   - a one-sentence **caption** (used by the CAP baseline),
   - the **events** in the window (`who | action | object | place`),
   - **yes/no confirmation** of each candidate state change from notebook 02 that falls in the window.
3. **Cleans the events:** checks the IDs, joins an event that continues into the next window, and **snaps** event boundaries to the nearest state change of the same object within ±1 s. The window times are kept too, so E2 can compare snapping on vs off.
4. **Marks state changes** as `confirmed` (the VLM said yes) or `uncertain` (no or not asked). Nothing is deleted.
5. **Derives the STAR pause and sham versions** from the original clips by shifting times (section 6).

**Two members, in parallel:** the same `SHARD` as in notebook 02. Each member needs only their own shard's 02 outputs.

**Runtime:** GPU (T4). Expect several hours per member, split over sessions. Each run of the main cell stops cleanly after 110 minutes; run it again to continue. A disconnect loses at most the last `SYNC_EVERY` videos.

**How results are saved:** as in 02. Files are written to the Colab disk, then uploaded every `SYNC_EVERY` videos as one zip pack in `outputs/cv/`, which is read back from Drive to verify it. If an upload cannot be verified, the notebook stops: do **not** restart the runtime then.

| Inside the packs (`outputs/cv/03_*.zip`) | Content |
| --- | --- |
| `events_raw/<id>.json` | per window: frames used, prompt inputs, the model's reply, parsed fields, seconds |
| `events/<id>.json` | cleaned events, state changes with their status, object state intervals |
| `captions/<id>.json` | one caption per window |
| `frames/<id>/w<start>.jpg` | the key frame (with overlays) of every window that produced an event |

In [1]:
# --- Parameters: the only cell to edit ---------------------------------------------------------
SHARD = 3                 # member A: 0, member B: 1 (same as in 02)
N_SHARDS = 4
GROUPS = ['dev', 'star', 'main']             # order of work; 'intentqa' is only needed for Tier 2
LIMIT_PER_GROUP = None    # e.g. 2 for a quick test run; None for the real run
VARIANT = 'main'          # 'main', or 'no_overlay' for the E2 ablation (STAR sequence clips, no boxes drawn)
BATCH = 2                 # windows per model call; lowered automatically if the GPU runs out of memory
SYNC_EVERY = 10           # upload results to Drive after this many videos (at most this many are redone after a crash)

In [2]:
# --- 0. Bootstrap --------------------------------------------------------------------------
import os, glob
ST_NOTEBOOK = '03_cv_events'
ST_QUIET = True
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')  # before torch loads
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import drive
    drive.mount('/content/drive')
    SETUP_NB = '/content/drive/MyDrive/ST-Reason/notebooks/00_setup_and_utils.ipynb'
elif os.path.exists('/kaggle/working'):
    SETUP_NB = (glob.glob('/kaggle/input/**/00_setup_and_utils.ipynb', recursive=True) or ['00_setup_and_utils.ipynb'])[0]
else:
    SETUP_NB = os.environ.get('ST_SETUP_NB', '00_setup_and_utils.ipynb')
%run -i $SETUP_NB

MOCK = os.environ.get('ST_MOCK_MODELS') == '1'      # local testing only: fake models, no GPU
assert SHARD in range(N_SHARDS), 'SHARD must be 0 .. N_SHARDS-1'
# each member appends to their own manifest, so two people never write the same file at once
MANIFEST_PATH = P['runs'] / f'manifest_shard{SHARD}.csv'
# Log to the Colab disk; a copy goes to Drive with every upload. (Appending to a file on Drive line by line
# makes Drive store many duplicate copies of it.)
DRIVE_LOG_PATH = LOG_PATH
LOG_PATH = P['local'] / 'logs' / LOG_PATH.name
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive
Runtime: colab | project folder: /content/drive/MyDrive/ST-Reason | local scratch: /content/st_local
config hash: 2a9e0e4f87


In [3]:
# --- Shared CV helpers (this cell is identical in 02 and 03) -------------------------------------
import pandas as pd, numpy as np, cv2
from PIL import Image, ImageDraw, ImageFont

CVC = CFG['cv']
CVC.setdefault('openvocab_fps', 1.0)          # pilot: YOLOE at 1 fps is enough and 5x cheaper
CVC.setdefault('stitch_min_cos', 0.80)
CVC.setdefault('state_fill_gap_s', 1.0)
CVC.setdefault('state_min_s', 0.6)
CVC.setdefault('event_window_s', CVC['window_s'])   # pilot: non-overlapping 3 s windows
# Per-video files are written to the fast Colab disk (LOCAL_CV) and uploaded to Drive in batches, as a
# few zip "packs" placed directly in outputs/cv/. Each pack is read back from Drive after upload to prove
# it arrived. (Writing thousands of small files to Drive can silently lose them, and new sub-folders made
# from Colab inside a folder shared from someone else's Drive can vanish, so none are created.)
LOCAL_CV = P['local'] / 'cv_out'
PACKS = P['cv']
PACK_GLOB = '0[23]*_shard*.zip'
CVD = {k: LOCAL_CV / k for k in ['frames_index', 'tracks', 'states', 'events_raw', 'events', 'captions', 'frames']}
for _d in CVD.values():
    _d.mkdir(parents=True, exist_ok=True)
_PACKED = set()            # paths (relative to LOCAL_CV) already safely inside a pack on Drive

def restore_from_packs(prefixes=None):
    """Unpack every pack on Drive into LOCAL_CV (files already there are skipped).
    prefixes: only these top-level folders, e.g. ['states', 'tracks']."""
    n_new, bad = 0, []
    for z in sorted(PACKS.glob(PACK_GLOB), reverse=True):   # newest first: a redone video's newest copy wins
        try:
            with zipfile.ZipFile(z) as zf:
                for m in zf.namelist():
                    _PACKED.add(m)
                    if prefixes and m.split('/')[0] not in prefixes:
                        continue
                    out = LOCAL_CV / m
                    if not out.exists():
                        out.parent.mkdir(parents=True, exist_ok=True)
                        with zf.open(m) as s, open(out, 'wb') as d:
                            shutil.copyfileobj(s, d)
                        n_new += 1
        except (zipfile.BadZipFile, OSError) as e:
            bad.append(z.name)
    if bad:
        log(f'{len(bad)} damaged packs skipped (an upload was cut off; their videos are redone): {bad[:3]}', 'WARN')
    log(f'restored {n_new} files from {len(list(PACKS.glob(PACK_GLOB)))} packs on Drive')

class DriveSyncError(BaseException):
    """Stops the whole loop (run_resumable only skips ordinary errors), so no more work piles up unsaved."""

def sync_to_drive(subdirs, tag):
    """Upload every local file in `subdirs` that is not yet on Drive, as one new pack, and verify it."""
    new = []
    for sd in subdirs:
        for p in (LOCAL_CV / sd).rglob('*'):
            if p.is_file() and not p.name.endswith('.tmp'):
                rel = str(p.relative_to(LOCAL_CV))
                if rel not in _PACKED:
                    new.append(rel)
    if not new:
        return 0
    name = f'{tag}_shard{SHARD}_{dt.datetime.now():%Y%m%d-%H%M%S}_{len(new):05d}.zip'
    tmp = P['local_tmp'] / name
    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for rel in new:
            zf.write(LOCAL_CV / rel, rel, compress_type=zipfile.ZIP_STORED if rel.endswith('.jpg') else zipfile.ZIP_DEFLATED)
    dest = PACKS / name
    for attempt in range(3):
        try:
            shutil.copyfile(tmp, dest)
            with zipfile.ZipFile(dest) as zf:                  # read back from Drive
                ok = len(zf.namelist()) == len(new) and zf.testzip() is None
            if ok:
                break
        except Exception as e:
            ok = False
            log(f'upload of {name} failed ({e.__class__.__name__}: {e}); retrying', 'WARN')
        time.sleep(5 * (attempt + 1))
    if not ok:
        raise DriveSyncError(f'Drive did not keep {name}. The results are still on this Colab disk: do NOT restart '
                           'the runtime. Check Drive space and that you can edit MyDrive/ST-Reason, then run the '
                           'sync cell again.')
    _PACKED.update(new)
    tmp.unlink(missing_ok=True)
    try:
        shutil.copyfile(LOG_PATH, DRIVE_LOG_PATH)          # one overwrite per upload, not one per line
    except Exception:
        pass
    log(f'uploaded {name}: {len(new)} files ({_gb(dest.stat().st_size) * 1024:.1f} MB), verified on Drive')
    return len(new)

restore_from_packs()

COCO = set(['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light',
    'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant',
    'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard',
    'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle',
    'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli',
    'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet',
    'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink',
    'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'])
# objects that other objects can rest "on"
SURFACES = {'dining table', 'table', 'desk', 'bed', 'couch', 'sofa', 'chair', 'bench', 'shelf', 'counter',
            'cabinet', 'toilet', 'sink', 'mat', 'rug', 'carpet', 'tray', 'stool', 'cushion', 'pillow'}

def save_compact(obj, path):
    """Atomic JSON write without indentation (CV files are large)."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    with open(tmp, 'w') as f:
        json.dump(obj, f, separators=(',', ':'), ensure_ascii=False, default=_json_default)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)

def decode_frames(path, fps_out):
    """Yield (t_seconds, BGR frame) at about fps_out, decoding once and streaming.
    02 and 03 both use this, so every stage sees exactly the same timestamps."""
    cap = cv2.VideoCapture(str(path))
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    step, i, next_t = 1.0 / fps_out, 0, 0.0
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        t = i / src_fps
        if t + 1e-6 >= next_t:
            yield round(t, 3), fr
            next_t += step
        i += 1
    cap.release()

def stratified_subset(df, n, seed):
    """Exactly n rows, allocated to question types in proportion to their counts (same as 01)."""
    n = min(n, len(df))
    counts = df.type.value_counts()
    alloc = (counts / counts.sum() * n).round().astype(int).clip(upper=counts)
    order, i = list(counts.index), 0
    while alloc.sum() != n:
        t = order[i % len(order)]
        if alloc.sum() < n and alloc[t] < counts[t]:
            alloc[t] += 1
        elif alloc.sum() > n and alloc[t] > 0:
            alloc[t] -= 1
        i += 1
    parts = [df[df.type == t].sample(n=int(k), random_state=seed) for t, k in alloc.items() if k > 0]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

def main_grid_eval():
    """The questions the main grid is actually run on. The pilot rule cut the grid to
    config data.main_grid_n (400). The subset keeps all 200 robustness questions and adds a
    stratified sample of the rest, so the robustness subset stays inside it. Frozen once saved."""
    path = P['subsets'] / 'main_grid_eval.csv'
    if path.exists():
        return pd.read_csv(path, dtype={'video_id': str})
    grid = pd.read_csv(P['subsets'] / 'main_grid.csv', dtype={'video_id': str})
    n = int(CFG['data'].get('main_grid_n') or (CFG.get('pilot_decisions') or {}).get('main_grid_n') or len(grid))
    if n >= len(grid):
        sub = grid
    else:
        rob = pd.read_csv(P['subsets'] / 'robustness.csv', dtype={'video_id': str})
        if n <= len(rob):
            keep = set(stratified_subset(rob, n, CFG['seed'] + 5).uid)
        else:
            rest = grid[~grid.uid.isin(set(rob.uid))]
            keep = set(rob.uid) | set(stratified_subset(rest, n - len(rob), CFG['seed'] + 5).uid)
        sub = grid[grid.uid.isin(keep)].reset_index(drop=True)
    if SHARD == 0 and ST_NOTEBOOK == '02_cv_detect_track':   # written once, by 02 only
        sub.to_csv(path, index=False)
        log(f'main_grid_eval.csv frozen: {len(sub)} questions, {sub.video_id.nunique()} videos')
    return sub

def main_grid_eval_hash(sub):
    return hashlib.sha1(sub.uid.astype(str).str.cat(sep='|').encode()).hexdigest()[:12]

_ZIP_INDEX = {}
def zip_index(name):
    """{video stem: member path} for a video zip on Drive (cached)."""
    if name not in _ZIP_INDEX:
        zp = P['data'] / name
        if not zp.exists():
            _ZIP_INDEX[name] = {}
        else:
            with zipfile.ZipFile(zp) as zf:
                _ZIP_INDEX[name] = {Path(n).stem: n for n in zf.namelist() if n.lower().endswith('.mp4')}
    return _ZIP_INDEX[name]

GROUP_ORDER = {'dev': 0, 'star': 1, 'main': 2, 'intentqa': 3}
def build_work_list():
    """Every video the CV pipeline must process, with its source zip, duration and shard.
    Shards are balanced by total video seconds, tier by tier, and are the same on every
    member's machine (deterministic), so nobody needs to coordinate."""
    meta = pd.read_csv(P['ann'] / 'video_meta.csv', dtype={'video_id': str}).set_index('video_id').duration_s
    rows, missing = [], {}
    nextqa_sets = [('dev', pd.read_csv(P['subsets'] / 'dev.csv', dtype={'video_id': str}), 'videos_nextqa_eval.zip', 1),
                   ('main', main_grid_eval(), 'videos_nextqa_eval.zip', 1),
                   ('intentqa', pd.read_csv(P['subsets'] / 'intentqa_eval.csv', dtype={'video_id': str}),
                    'videos_intentqa_eval.zip', 2)]
    for group, df, zname, tier in nextqa_sets:
        idx = zip_index(zname)
        for v in sorted(df.video_id.unique()):
            if v not in idx:
                missing.setdefault(group, []).append(v); continue
            rows.append({'key': v, 'group': group, 'tier': tier, 'zip': zname, 'member': idx[v],
                         'duration_s': meta.get(v, np.nan)})
    man = P['data'] / 'star' / 'clip_manifest.csv'
    if man.exists():
        M = pd.read_csv(man)
        idx = zip_index('videos_star_eval.zip')
        for r in M[M.version == 'orig'].itertuples(index=False):
            if r.clip_id not in idx:
                missing.setdefault('star', []).append(r.clip_id); continue
            rows.append({'key': r.clip_id, 'group': 'star', 'tier': 1, 'zip': 'videos_star_eval.zip',
                         'member': idx[r.clip_id], 'duration_s': r.n_frames_src / r.fps})
    else:
        log('data/star/clip_manifest.csv not found: run 01b_star first. STAR clips are skipped for now.', 'WARN')
    W = pd.DataFrame(rows).drop_duplicates('key').reset_index(drop=True)
    W['duration_s'] = W.duration_s.fillna(W.duration_s.median()).astype(float)
    W['shard'] = -1
    for tier in sorted(W.tier.unique()):              # greedy: longest video to the least-loaded shard
        load = [0.0] * N_SHARDS
        sub = W[W.tier == tier].sort_values(['duration_s', 'key'], ascending=[False, True])
        for i, d in zip(sub.index, sub.duration_s):
            s = int(np.argmin(load)); W.at[i, 'shard'] = s; load[s] += d
    W['order'] = W.group.map(GROUP_ORDER)
    W = W.sort_values(['tier', 'order', 'key']).drop(columns='order').reset_index(drop=True)
    for g, v in missing.items():
        log(f'{len(v)} {g} videos are not in their zip and are skipped: {v[:5]}', 'WARN')
    return W

class VideoSource:
    """Extracts one video at a time from its Drive zip to the Colab disk, and deletes it after use."""
    def __init__(self):
        self.dir = P['local_videos'] / 'cv'
        self.dir.mkdir(parents=True, exist_ok=True)
        self.zf = {}
    def get(self, row):
        out = self.dir / f'{row.key}.mp4'
        if not out.exists():
            if row.zip not in self.zf:
                self.zf[row.zip] = zipfile.ZipFile(P['data'] / row.zip)
            part = out.with_suffix('.part')
            with self.zf[row.zip].open(row.member) as s, open(part, 'wb') as d:
                shutil.copyfileobj(s, d)
            os.replace(part, out)
        return out
    def drop(self, row):
        (self.dir / f'{row.key}.mp4').unlink(missing_ok=True)

def select_work(W, groups, limit=None):
    """This member's videos, in processing order (Tier 1 first)."""
    mine = W[(W.shard == SHARD) & W.group.isin(groups)]
    if limit:
        mine = mine.groupby('group', sort=False).head(limit)
    mine = mine.assign(_o=mine.group.map({g: i for i, g in enumerate(groups)}))
    return mine.sort_values(['tier', '_o', 'key']).drop(columns='_o').reset_index(drop=True)

def delete_model_cache(*repo_ids):
    """Free Colab disk: remove downloaded Hugging Face weights that are no longer needed."""
    hub = Path(os.environ.get('HF_HOME', Path.home() / '.cache' / 'huggingface')) / 'hub'
    for rid in repo_ids:
        d = hub / ('models--' + rid.replace('/', '--'))
        if d.exists():
            shutil.rmtree(d, ignore_errors=True)
            log(f'deleted cached weights: {rid}')

WORK = build_work_list()
MGE_HASH = main_grid_eval_hash(main_grid_eval())
print('main_grid_eval hash:', MGE_HASH)
summary = WORK.pivot_table(index='group', columns='shard', values='duration_s', aggfunc=['count', 'sum'])
summary.columns = [f'{a}_shard{b}' for a, b in summary.columns]
print(summary.round(0).astype(int).to_string())

[10:27:31] INFO  restored 6983 files from 77 packs on Drive
main_grid_eval hash: e83b2604bc79
          count_shard0  count_shard1  count_shard2  count_shard3  sum_shard0  sum_shard1  sum_shard2  sum_shard3
group                                                                                                           
dev                 22            21            24            25         887         933         946        1115
intentqa            55            56            56            57        2648        2658        2648        2658
main                44            40            49            42        1752        1627        1927        1592
star                91            96            84            89        1245        1324        1009        1174


## 1. Which videos are ready

A video is ready when notebook 02 has saved its states file. For `VARIANT = 'no_overlay'` (E2), only the STAR clips of sequence questions are used, and outputs go to separate folders (`events_no_overlay/`, …), so the main outputs are never touched.

In [4]:
assert VARIANT in ('main', 'no_overlay')
OVERLAY = VARIANT == 'main'
SFX = '' if VARIANT == 'main' else f'_{VARIANT}'
RAW_DIR, EV_DIR, CAP_DIR = LOCAL_CV / f'events_raw{SFX}', LOCAL_CV / f'events{SFX}', LOCAL_CV / f'captions{SFX}'
SUBDIRS_03 = [RAW_DIR.name, EV_DIR.name, CAP_DIR.name] + (['frames'] if VARIANT == 'main' else [])
TAG = '03' if VARIANT == 'main' else f'03{SFX}'
for d in (RAW_DIR, EV_DIR, CAP_DIR):
    d.mkdir(parents=True, exist_ok=True)

MY = select_work(WORK, GROUPS, LIMIT_PER_GROUP)
if VARIANT == 'no_overlay':
    star_main = pd.read_csv(P['subsets'] / 'star_main.csv')
    seq_keys = {f'{u}__orig' for u in star_main[star_main.type == 'Sequence'].uid}
    MY = select_work(WORK[WORK.key.isin(seq_keys)], ['star'], LIMIT_PER_GROUP)
ready = done_ids(CVD['states'])

def done_raw():
    """Videos whose raw replies come from the current prompt version (older ones are redone)."""
    out = set()
    for f in RAW_DIR.glob('*.json'):
        try:
            if load_json(f).get('prompt_version') == PROMPT_VERSION_NOW:
                out.add(f.stem)
        except Exception:
            pass
    return out
PROMPT_VERSION_NOW = 'lines-v3'   # v3: worn items not outlined; no appear/disappear events
READY = MY[MY.key.isin(ready)].reset_index(drop=True)
not_ready = MY[~MY.key.isin(ready)]
done = done_raw()
print(f'shard {SHARD}, variant {VARIANT}: {len(MY)} videos, {len(READY)} ready (02 done), '
      f'{len(set(READY.key) & done)} already have events')
if len(not_ready):
    log(f'{len(not_ready)} videos of this shard are not through notebook 02 yet; they will be picked up '
        'when this notebook is run again after 02', 'WARN')

shard 3, variant main: 156 videos, 156 ready (02 done), 20 already have events


## 2. Model

Qwen3-VL-4B-Instruct in fp16, as decided by the pilot. Windows are sent in batches of `BATCH`. If the GPU runs out of memory, the batch is halved and retried. If batching itself fails in this transformers version, the notebook falls back to one window per call and says so.

In [5]:
if not MOCK:
    pip_install('transformers>=4.57.0', 'accelerate>=1.0')
VLM_SPEC = dict(CFG['models']['vlm_events'])
MAX_NEW = 90               # short caption + at most 4 event lines + CHECK lines

def load_vlm(spec):
    import torch
    from transformers import AutoModelForImageTextToText, AutoProcessor
    # whole model on the GPU, nothing silently moved to the CPU (that made every call ~10x slower)
    kw = {'device_map': {'': 0}, 'low_cpu_mem_usage': True, 'revision': spec.get('revision', 'main')}
    if spec.get('quant') == 'nf4':
        from transformers import BitsAndBytesConfig
        kw['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                                       bnb_4bit_compute_dtype=torch.float16)
    else:
        kw['dtype'] = torch.float16                             # the T4 has no bf16
    try:
        model = AutoModelForImageTextToText.from_pretrained(spec['id'], **kw).eval()
    except TypeError:                                           # older transformers: torch_dtype
        kw['torch_dtype'] = kw.pop('dtype', torch.float16)
        model = AutoModelForImageTextToText.from_pretrained(spec['id'], **kw).eval()
    devices = {str(p.device) for p in model.parameters()}
    if devices != {'cuda:0'} or model.dtype != torch.float16:
        raise RuntimeError(f'model is not fully on the GPU in fp16 (devices {devices}, dtype {model.dtype}). '
                           'Use Runtime -> Disconnect and delete runtime, then Run all once.')
    proc = AutoProcessor.from_pretrained(spec['id'], revision=spec.get('revision', 'main'))
    proc.tokenizer.padding_side = 'left'                       # required for batched generation
    return model, proc

def vlm_generate_batch(model, proc, items, max_new_tokens):
    """items: [(images, prompt)] -> [reply text], one call for the whole batch."""
    import torch
    msgs = [[{'role': 'user', 'content': [*({'type': 'image', 'image': im} for im in ims),
                                          {'type': 'text', 'text': pr}]}] for ims, pr in items]
    inputs = proc.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_dict=True,
                                      return_tensors='pt', padding=True).to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return proc.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)

if MOCK:   # local testing only: a canned reply that uses the IDs in the prompt
    def load_vlm(spec):
        return None, None
    def vlm_generate_batch(model, proc, items, max_new_tokens):
        outs = []
        for ims, pr in items:
            m = re.search(r'corner: ([^\n]+?)\.\n', pr)
            ids = m.group(1).split(', ') if m else []
            who = next((i for i in ids if i.startswith('person')), 'man')
            obj = next((i for i in ids if not i.startswith('person')), 'cup')
            n = len(re.findall(r'^\d+\. ', pr, flags=re.M))
            outs.append('\n'.join(['CAPTION: A person handles an object.', f'EVENT: {who} | picks up | {obj} | kitchen']
                                  + [f'CHECK {i}: {"yes" if i % 2 else "no"}' for i in range(1, n + 1)]))
        return outs

STATE = {'batch': BATCH, 'batching_ok': True}
if MOCK:
    OOM_ERROR = MemoryError
else:
    import torch
    OOM_ERROR = torch.cuda.OutOfMemoryError
def generate_all(items):
    """Replies for all items, in order, with OOM backoff and a one-window fallback."""
    outs, i = [None] * len(items), 0
    while i < len(items):
        cap = STATE['batch'] if STATE['batching_ok'] else 1
        chunk = items[i:i + cap]
        try:
            res = vlm_generate_batch(MODEL, PROC, chunk, MAX_NEW)
            outs[i:i + len(chunk)] = res
            i += len(chunk)
        except OOM_ERROR:
            free_memory()
            if len(chunk) == 1:
                raise
            STATE['batch'] = max(1, len(chunk) // 2)
            log(f'GPU memory full at batch {len(chunk)}: continuing with batch {STATE["batch"]}', 'WARN')
        except Exception as e:
            if len(chunk) > 1 and STATE['batching_ok']:
                STATE['batching_ok'] = False
                log(f'batched generation failed ({e.__class__.__name__}: {str(e)[:150]}); using one window per call', 'WARN')
            else:
                raise
    return outs

if not MOCK:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()                        # hides a harmless warning printed on every call
if len(READY):
    if 'MODEL' in globals() and MODEL is not None:          # this cell was run before: reuse, never load twice
        print('model already loaded: reusing it')
    else:
        if not MOCK:
            import torch
            free_memory()
            used = torch.cuda.memory_allocated() / 1e9
            if used > 1.0:
                raise RuntimeError(f'{used:.1f} GB of GPU memory is already in use before loading. '
                                   'Use Runtime -> Disconnect and delete runtime, then Run all once.')
        MODEL, PROC = load_vlm(VLM_SPEC)
    if not MOCK:
        print(f'loaded {VLM_SPEC["id"]} on the GPU in fp16 | GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

[10:27:40] INFO  pip install transformers>=4.57.0 accelerate>=1.0


config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/64.7k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

loaded Qwen/Qwen3-VL-4B-Instruct on the GPU in fp16 | GPU memory in use: 8.9 GB


## 3. Windows, frames and the prompt

- **Windows:** 0–3 s, 3–6 s, … to the end of the video. A last piece shorter than 1 s is joined to the window before it.
- **Frames:** 4 per window, evenly spaced among the 5 fps timestamps saved by 02, resized to about 448×448.
- **Overlays:** each tracked person or object gets a thin coloured outline and its ID in the corner, with a fixed colour per ID. Not outlined: tracks flagged `short`, parts of the scene found by the open vocabulary (wall, floor, door, window, …), whose boxes would cover the frame, and worn items (hat, shirt, glasses, …), which otherwise produce "hat appears / disappears" events. Scene parts are also never sent for state checks, and are marked `scene` in the output.
- **Prompt:** asks for one caption line, one `EVENT` line per action, and one `CHECK` line per candidate state change in the window (at most 6, held/on changes first).

The reply format is plain lines, not JSON. The pilot's JSON replies were long and 10% failed to parse; the line format is shorter and a broken line only loses that line.

In [6]:
import zlib
PROMPT_VERSION = PROMPT_VERSION_NOW   # v2: 'none' instead of '-', at most 4 events, short caption
MAX_CHECKS, MAX_IDS = 6, 15
try:
    FONT = ImageFont.load_default(size=14)
except TypeError:                                   # older Pillow
    FONT = ImageFont.load_default()
PALETTE = [(230, 25, 75), (60, 180, 75), (0, 130, 200), (245, 130, 48), (145, 30, 180), (70, 240, 240),
           (240, 50, 230), (210, 245, 60), (0, 128, 128), (170, 110, 40)]

def windows_for(duration, win):
    out, s = [], 0.0
    while s < duration - 1e-6:
        out.append([round(s, 2), round(min(s + win, duration), 2)]); s += win
    if len(out) > 1 and out[-1][1] - out[-1][0] < 1.0:
        out[-2][1] = out[-1][1]; out.pop()
    return [tuple(w) for w in out]

def pick_times(times, ws, we, k, last):
    ts = [t for t in times if ws - 1e-6 <= t < we - 1e-6 or (last and ws <= t <= we + 1e-6)]
    if not ts:
        return []
    idx = sorted(set(np.linspace(0, len(ts) - 1, min(k, len(ts))).round().astype(int).tolist()))
    return [ts[i] for i in idx]

def to_pil(frame_bgr, max_pixels):
    im = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    w, h = im.size
    scale = min(1.0, (max_pixels / (w * h)) ** 0.5)
    return im.resize((max(28, int(w * scale)), max(28, int(h * scale)))) if scale < 1 else im

def draw_tracks(im, boxes, sx, sy):
    d = ImageDraw.Draw(im)
    for b in boxes:
        x1, y1, x2, y2 = b['xyxy'][0] * sx, b['xyxy'][1] * sy, b['xyxy'][2] * sx, b['xyxy'][3] * sy
        c = PALETTE[zlib.crc32(b['id'].encode()) % len(PALETTE)]
        d.rectangle([x1, y1, x2, y2], outline=c, width=2)
        tx, ty = x1 + 2, max(0, y1 + 1)
        try:
            l, t_, r, bt = d.textbbox((tx, ty), b['id'], font=FONT)
            d.rectangle([l - 1, t_ - 1, r + 1, bt + 1], fill=(0, 0, 0))
        except Exception:
            pass
        d.text((tx, ty), b['id'], fill=c, font=FONT)
    return im

STATE_TEXT = {'held_by': 'held by {}', 'on': 'on {}', 'near': 'near {}', 'absent': 'not visible',
              'visible': 'visible, not held'}
def state_text(st, tg):
    return STATE_TEXT.get(st, st).format(tg)

def make_prompt(ws, we, k, ids, checks):
    head = f'These {k} frames are from {ws:.1f}s to {we:.1f}s of a video, in time order.'
    if OVERLAY and ids:
        head += f' People and objects are outlined, with their IDs written in the corner: {", ".join(ids)}.'
        who = 'the ID from the list when the person or object is outlined, otherwise a short name'
    else:
        who = 'a short name such as "man", "girl" or "cup"'
    lines = [head,
             'Describe what happens. Reply only in this line format:',
             'CAPTION: <one short sentence, at most 15 words>',
             'EVENT: <who> | <action, a short verb phrase> | <object, or none> | <place, or none>',
             f'Write at most 4 EVENT lines, the most important actions first, one action per line. '
             'Only report actions that people or animals do; do not report things appearing or disappearing. '
             f'For people and objects use {who}. If nobody does anything, write: EVENT: none']
    if checks:
        lines.append('Then look at each possible change below and answer one line each, as: CHECK <number>: yes or no')
        for i, c in enumerate(checks, 1):
            lines.append(f'{i}. {c["obj"]}: {state_text(*c["from"])} -> {state_text(*c["to"])} (around {c["t"]:.1f}s)')
    return '\n'.join(lines)

NULLS = {'', '-', 'none', 'null', 'n/a', 'na', 'nothing', 'nobody', 'unknown'}
def parse_reply(txt):
    cap, events, checks, saw_none = None, [], {}, False
    for line in (txt or '').splitlines():
        l = re.sub(r'[*_`#]{2,}|^[*\-•>#\s]+', '', line).strip()      # drop markdown bold/bullets
        m = re.match(r'(?i)caption\s*:\s*(.+)', l)
        if m:
            cap = m.group(1).strip(); continue
        m = re.match(r'(?i)event\s*:\s*(.*)', l)
        if m:
            body = m.group(1).strip()
            if body.lower().rstrip('.') in NULLS:
                saw_none = True; continue
            parts = [p.strip() for p in body.split('|')] + [''] * 4
            if parts[1] and parts[1].lower() not in NULLS:
                nz = lambda x: (lambda y: None if y.lower().strip('.') in NULLS else y)(x.strip().lstrip('->').strip())
                events.append({'who': nz(parts[0]), 'verb': parts[1].strip('.').lower(),
                               'object': nz(parts[2]), 'place': nz(parts[3])})
            continue
        m = re.match(r'(?i)check\s*(\d+)\s*:\s*(yes|no)', l)
        if m:
            checks[m.group(1)] = m.group(2).lower() == 'yes'
    return {'caption': cap, 'events': events, 'check_answers': checks,
            'parsed': cap is not None or bool(events) or saw_none}

# parts of the scene, not things people handle: no outline (they would cover the frame) and no state checks
SCENE = {'wall', 'floor', 'ceiling', 'door', 'window', 'stairs', 'staircase', 'curtain', 'carpet', 'rug', 'mat',
         'grass', 'tree', 'trees', 'sky', 'water', 'sand', 'rock', 'stone', 'road', 'fence', 'gate', 'pole',
         'light', 'lamp', 'shelf', 'cabinet', 'building', 'room', 'ground', 'pool', 'snow', 'field'}

# worn items: always inside a person's box; outlined, they produce 'hat appears / disappears' noise
WEARABLES = {'glasses', 'sunglasses', 'hat', 'cap', 'helmet', 'shoe', 'shoes', 'sock', 'socks', 'shirt', 'jacket',
             'coat', 'dress', 'scarf', 'glove', 'gloves', 'ring', 'necklace', 'bracelet', 'watch', 'tie', 'mask',
             'costume', 'diaper', 'bib', 'pants', 'shorts', 'skirt', 'boot', 'boots'}

PRIORITY = {'held_by': 0, 'on': 0, 'absent': 1, 'near': 2, 'visible': 3}
def window_checks(changes, ws, we):
    c = [x for x in changes if ws - 1e-6 <= x['t'] < we - 1e-6]
    c.sort(key=lambda x: (min(PRIORITY.get(x['from'][0], 3), PRIORITY.get(x['to'][0], 3)), x['t']))
    return sorted(c[:MAX_CHECKS], key=lambda x: x['t'])

## 4. Cleaning: IDs, joining, snapping, confirmation

- **IDs.** A reply like `Person 1` or `cup3` is normalised to `person_1` / `cup_3`. It is kept as an ID if it was drawn in that window (`in_window`) or elsewhere in the video (`elsewhere`). Otherwise the text is kept as a name, with no ID.
- **Joining.** The same (who, action, object) in two adjacent windows is one event. If a window without it comes between, they stay two events, so a repeated action is not merged away.
- **Snapping.** The start moves to the nearest candidate state change of the same object within ±1 s, and the end likewise, when one exists. `t0_win`/`t1_win` keep the window times for the snapping-off condition of E2.
- **Confirmation.** A change answered `yes` is `confirmed`. Any other change, including one never asked about, is `uncertain`. Each state interval carries the status of the change that starts it; an object's first interval is `initial`.

In [7]:
def norm_ref(x, window_ids, known):
    """(id or None, original text or None, how it matched)."""
    if not x:
        return None, None, None
    raw = x.strip().strip('"\'.,')
    if raw.lower() in NULLS:
        return None, None, None
    c = re.sub(r'[\s\-]+', '_', raw.lower())
    c = re.sub(r'([a-z])(\d+)$', r'\1_\2', c)
    if c in window_ids:
        return c, raw, 'in_window'
    if c in known:
        return c, raw, 'elsewhere'
    return None, raw, 'name'

def postprocess(key):
    raw = load_json(RAW_DIR / f'{key}.json')
    st = load_json(CVD['states'] / f'{key}.json')
    known = set(raw.get('known_ids', []))
    tol = CVC['snap_tolerance_s']

    # 1. state changes: confirmed / uncertain
    answers = {}
    for w in raw['windows']:
        for i, cid in enumerate(w['checks'], 1):
            a = w['check_answers'].get(str(i))
            if a is not None:
                answers[cid] = a
    asked = {x for w in raw['windows'] for x in w['checks']}
    changes = []
    for c in st['changes']:
        a = answers.get(c['id'])
        changes.append({**c, 'vlm_answer': None if a is None else ('yes' if a else 'no'), 'asked': c['id'] in asked,
                        'status': 'confirmed' if a is True else 'uncertain'})
    status_at = {(c['obj'], c['t']): c['status'] for c in changes}
    objects = {}
    for oid, o in st['objects'].items():
        ivs = []
        for j, iv in enumerate(o['intervals']):
            ivs.append({**iv, 'status': 'initial' if j == 0 else status_at.get((oid, iv['t0']), 'uncertain')})
        objects[oid] = {**o, 'intervals': ivs, 'scene': o['cls'] in SCENE}

    # 2. events: IDs, then join across adjacent windows
    events, open_ = [], {}
    for w in raw['windows']:
        wids = set(w['ids'])
        seen = set()
        for e in w['events']:
            a_id, a_txt, a_how = norm_ref(e['who'], wids, known)
            o_id, o_txt, o_how = norm_ref(e['object'], wids, known)
            k = (a_id or (a_txt or '').lower(), e['verb'], o_id or (o_txt or '').lower())
            if k in seen:
                continue
            seen.add(k)
            prev = open_.get(k)
            if prev is not None and abs(prev['t1_win'] - w['ws']) < 1e-6:
                prev['t1_win'] = w['we']; prev['windows'].append(w['ws'])
                if w.get('frame') and not prev.get('frame'):
                    prev['frame'] = w['frame']
            else:
                ev = {'actor': a_id, 'actor_text': a_txt, 'actor_match': a_how, 'verb': e['verb'],
                      'object': o_id, 'object_text': o_txt, 'object_match': o_how, 'location': e['place'],
                      't0_win': w['ws'], 't1_win': w['we'], 'windows': [w['ws']], 'frame': w.get('frame')}
                events.append(ev); open_[k] = ev

    # 3. snapping to state changes of the same object
    ch_t = {}
    for c in changes:
        ch_t.setdefault(c['obj'], []).append(c['t'])
    def nearest(obj, t, lo=None):
        cand = [x for x in ch_t.get(obj, []) if abs(x - t) <= tol + 1e-6 and (lo is None or x > lo + 0.2)]
        return min(cand, key=lambda x: abs(x - t)) if cand else None
    for e in events:
        s0 = nearest(e['object'], e['t0_win']) if e['object'] else None
        e['t0'] = s0 if s0 is not None else e['t0_win']
        s1 = nearest(e['object'], e['t1_win'], lo=e['t0']) if e['object'] else None
        e['t1'] = s1 if s1 is not None else e['t1_win']
        e['snapped'] = [s0 is not None, s1 is not None]
    events.sort(key=lambda e: (e['t0'], e['t1']))
    for i, e in enumerate(events, 1):
        e['eid'] = f'ev{i}'

    W = raw['windows']
    stats = {'windows': len(W), 'parsed': sum(w['parsed'] for w in W), 'events': len(events),
             'event_lines': sum(len(w['events']) for w in W),
             'actor_id': sum(e['actor'] is not None for e in events), 'object_id': sum(e['object'] is not None for e in events),
             'snapped_start': sum(e['snapped'][0] for e in events), 'changes': len(changes),
             'asked': sum(c['asked'] for c in changes), 'confirmed': sum(c['status'] == 'confirmed' for c in changes),
             'seconds': round(sum(w['seconds'] for w in W), 1)}
    save_compact({'key': key, 'variant': VARIANT, 'model': raw['model'], 'prompt_version': raw['prompt_version'],
                  'duration_s': st['duration_s'], 'events': events, 'changes': changes, 'objects': objects,
                  'stats': stats}, EV_DIR / f'{key}.json')
    save_compact({'key': key, 'duration_s': st['duration_s'],
                  'captions': [{'ws': w['ws'], 'we': w['we'], 'text': w['caption']} for w in W]}, CAP_DIR / f'{key}.json')
    return stats

## 5. Run

For each ready video: decode once at 5 fps, keep only the frames the windows need, build the prompts, generate in batches, save the raw replies (this marks the video as done), then clean and save. The local copy of the video is deleted afterwards.

In [8]:
SRC = VideoSource()
RUN = Run(params={'step': 'cv_events', 'variant': VARIANT, 'shard': SHARD, 'n_shards': N_SHARDS, 'groups': GROUPS,
                  'limit': LIMIT_PER_GROUP, 'model': VLM_SPEC['id'], 'prompt': PROMPT_VERSION, 'batch': BATCH,
                  'window_s': CVC['event_window_s'], 'frames_per_window': CVC['frames_per_window']},
          items_total=len(READY))
SESSION = []

def process(row):
    t_start = time.time()
    fi = load_json(CVD['frames_index'] / f'{row.key}.json')
    trk = load_json(CVD['tracks'] / f'{row.key}.json')
    st = load_json(CVD['states'] / f'{row.key}.json')
    good = {k for k, v in trk['tracks'].items()
            if not v['short'] and v['cls'] not in SCENE and v['cls'] not in WEARABLES}
    cand_changes = [c for c in st['changes'] if c['obj'] in good]
    boxes_at = {round(f['t'], 3): [b for b in f['boxes'] if b['id'] in good] for f in trk['frames']}
    wins = windows_for(fi['duration_s'], CVC['event_window_s'])
    picks = [pick_times(fi['times'], ws, we, CVC['frames_per_window'], i == len(wins) - 1) for i, (ws, we) in enumerate(wins)]
    need = {round(t, 3) for p in picks for t in p}
    frames = {}
    for t, fr in decode_frames(SRC.get(row), CVC['decode_fps']):
        rt = round(t, 3)
        if rt in need:
            frames[rt] = (to_pil(fr, CVC['vlm_max_pixels']), fr.shape[1], fr.shape[0])
    items, meta = [], []
    for (ws, we), ts in zip(wins, picks):
        ts = [round(t, 3) for t in ts if round(t, 3) in frames]
        if not ts:
            continue
        ims, ids = [], []
        for t in ts:
            im, w0, h0 = frames[t]
            im = im.copy()
            bx = boxes_at.get(t, [])
            if OVERLAY:
                draw_tracks(im, bx, im.size[0] / w0, im.size[1] / h0)
            ims.append(im)
            ids += [b['id'] for b in bx if b['id'] not in ids]
        ids = sorted(ids, key=lambda i: (not i.startswith('person'), i))[:MAX_IDS]
        checks = window_checks(cand_changes, ws, we) if OVERLAY else []
        items.append((ims, make_prompt(ws, we, len(ims), ids, checks)))
        meta.append({'ws': ws, 'we': we, 'frames': ts, 'ids': ids, 'checks': [c['id'] for c in checks]})
    del frames
    t0 = time.time()
    replies = generate_all(items) if items else []
    per = (time.time() - t0) / max(1, len(items))
    recs = []
    for (ims, prompt), m, rep in zip(items, meta, replies):
        p = parse_reply(rep)
        rec = {**m, 'reply': rep, **p, 'seconds': round(per, 2)}
        if VARIANT == 'main' and p['events']:
            fp = CVD['frames'] / row.key / f'w{m["ws"]:07.2f}.jpg'
            fp.parent.mkdir(parents=True, exist_ok=True)
            ims[len(ims) // 2].save(fp, quality=80)
            rec['frame'] = str(fp.relative_to(LOCAL_CV))      # path inside the packs
        recs.append(rec)
    save_compact({'key': row.key, 'variant': VARIANT, 'model': VLM_SPEC['id'], 'prompt_version': PROMPT_VERSION,
                  'overlay': OVERLAY, 'known_ids': sorted(trk['tracks']), 'windows': recs,
                  'example_prompt': items[0][1] if items else None}, RAW_DIR / f'{row.key}.json')
    stats = postprocess(row.key)
    SRC.drop(row)
    SESSION.append({'key': row.key, **stats, 'wall_s': round(time.time() - t_start, 1)})

N_SINCE_SYNC = [0]
def process_and_sync(row):
    process(row)
    N_SINCE_SYNC[0] += 1
    if N_SINCE_SYNC[0] >= SYNC_EVERY:
        sync_to_drive(SUBDIRS_03, TAG)
        N_SINCE_SYNC[0] = 0

try:
    res = run_resumable(list(READY.itertuples(index=False)), key=lambda r: r.key, fn=process_and_sync,
                        done=done_raw(), desc=f'events ({VARIANT})', progress_every=5)
    print(res)
    # clean any video whose raw replies were saved but whose cleaning was interrupted
    for k in sorted(done_raw() & set(READY.key) - done_ids(EV_DIR)):
        postprocess(k)
finally:
    sync_to_drive(SUBDIRS_03, TAG)             # always upload what is done, even after an error

[10:30:00] INFO  run started: 03_cv_events-20261017-102959-c39c30
[10:30:00] INFO  events (main): 156 total, 20 already done, 136 to do
[10:33:11] INFO  events (main): 5/136 | 38.2s each | ETA 83.3 min
[10:33:53] INFO  uploaded 03_shard3_20261017-103353_00051.zip: 51 files (0.0 MB), verified on Drive
[10:33:53] INFO  events (main): 10/136 | 23.3s each | ETA 49.0 min
[10:34:23] INFO  events (main): 15/136 | 17.6s each | ETA 35.4 min
[10:35:05] INFO  uploaded 03_shard3_20261017-103505_00053.zip: 53 files (0.0 MB), verified on Drive
[10:35:05] INFO  events (main): 20/136 | 15.3s each | ETA 29.5 min
[10:35:31] INFO  events (main): 25/136 | 13.3s each | ETA 24.5 min
[10:36:08] INFO  uploaded 03_shard3_20261017-103608_00052.zip: 52 files (0.0 MB), verified on Drive
[10:36:08] INFO  events (main): 30/136 | 12.3s each | ETA 21.7 min
[10:37:07] INFO  events (main): 35/136 | 12.2s each | ETA 20.5 min
[10:38:36] INFO  uploaded 03_shard3_20261017-103836_00069.zip: 69 files (0.0 MB), verified on Dr

## 6. STAR pause and sham versions (derived, no extra GPU time)

The pause and sham clips differ from the original only by a frozen frame inserted at one point. Their CV outputs are therefore derived from the original clip's outputs: every time at or after the insertion point moves later by the pause length, and an event or state interval that spans the insertion point is lengthened by it.

This keeps perception identical across the five versions, so the H2 comparison changes only the elapsed time between the two actions, which is the variable under test. Re-running the VLM on frozen frames would add noise unrelated to the gap and about 8 GPU hours. **Report this in the paper's method section.**

Each member derives the versions of their own shard's original clips. A version whose original is not done yet is derived on a later run.

In [9]:
def derive_gap_versions():
    man = P['data'] / 'star' / 'clip_manifest.csv'
    if VARIANT != 'main' or not man.exists():
        return {'derived': 0, 'pending': 0}
    M = pd.read_csv(man)
    mine = set(WORK[(WORK.shard == SHARD) & (WORK.group == 'star')].key)
    n_new, pending = 0, 0
    for r in M[M.version != 'orig'].itertuples(index=False):
        src = f'{r.uid}__orig'
        if src not in mine or (EV_DIR / f'{r.clip_id}.json').exists():
            continue
        if not (EV_DIR / f'{src}.json').exists():
            pending += 1; continue
        at, secs = float(r.insert_at_s), float(r.pause_s)
        s0 = lambda t: round(t + secs, 2) if t >= at - 1e-9 else t        # starts at/after the insert move
        s1 = lambda t: round(t + secs, 2) if t > at + 1e-9 else t         # ends after the insert move
        ev = load_json(EV_DIR / f'{src}.json')
        for e in ev['events']:
            e['t0'], e['t0_win'] = s0(e['t0']), s0(e['t0_win'])
            e['t1'], e['t1_win'] = s1(e['t1']), s1(e['t1_win'])
            e['windows'] = [s0(w) for w in e['windows']]
        for c in ev['changes']:
            c['t'] = s0(c['t'])
        for o in ev['objects'].values():
            for iv in o['intervals']:
                iv['t0'], iv['t1'] = s0(iv['t0']), s1(iv['t1'])
        ev.update(key=r.clip_id, duration_s=round(ev['duration_s'] + secs, 3), derived_from=src,
                  insert_at_s=at, pause_s=secs)
        cap = load_json(CAP_DIR / f'{src}.json')
        for c in cap['captions']:
            c['ws'], c['we'] = s0(c['ws']), s1(c['we'])
        cap.update(key=r.clip_id, duration_s=ev['duration_s'], derived_from=src)
        save_compact(cap, CAP_DIR / f'{r.clip_id}.json')
        save_compact(ev, EV_DIR / f'{r.clip_id}.json')           # written last: marks the version as done
        n_new += 1
    return {'derived': n_new, 'pending': pending}

GAP = derive_gap_versions()
sync_to_drive(SUBDIRS_03, TAG)
print('STAR pause/sham versions:', GAP)

STAR pause/sham versions: {'derived': 0, 'pending': 0}


## 7. Check and record

Quality numbers for this member's videos so far. Things to look at:
- **parse rate** should be above 0.95; if it is much lower, send the example replies printed below.
- **events per window** around 1–3 is normal.
- **object IDs used**: the share of events whose object is a tracked ID. This shows whether the overlays are being used.
- **confirmed**: the share of asked state changes the VLM confirmed.

In [10]:
GAP = globals().get('GAP', {})                 # section 6 may not have run yet
sync_to_drive(SUBDIRS_03, TAG)                 # safe to run again any time
on_drive = {Path(r).stem for r in _PACKED if r.startswith(EV_DIR.name + '/')}
EVS = [load_json(EV_DIR / f'{k}.json') for k in sorted(on_drive & set(READY.key))]
if EVS:
    S = pd.DataFrame([e['stats'] for e in EVS]).sum()
    report = {
        'videos_done': len(EVS), 'videos_in_shard': len(MY),
        'windows': int(S.windows), 'parse_rate': round(S.parsed / max(S.windows, 1), 3),
        's_per_window': round(S.seconds / max(S.windows, 1), 2),
        'events_per_window': round(S.event_lines / max(S.windows, 1), 2),
        'events_after_joining': int(S.events),
        'actor_id_share': round(S.actor_id / max(S.events, 1), 3),
        'object_id_share': round(S.object_id / max(S.events, 1), 3),
        'snapped_start_share': round(S.snapped_start / max(S.events, 1), 3),
        'changes_asked_share': round(S.asked / max(S.changes, 1), 3),
        'confirmed_of_asked': round(S.confirmed / max(S.asked, 1), 3),
        'star_gap_versions': GAP, 'variant': VARIANT, 'batch_final': STATE['batch'],
        'batching_ok': STATE['batching_ok'], 'updated': dt.datetime.now().isoformat(timespec='seconds')}
    save_json(report, P['cv'] / f'progress_03_{VARIANT}_shard{SHARD}.json')
    print(json.dumps(report, indent=1))
    ex = load_json(RAW_DIR / f'{EVS[0]["key"]}.json')
    print('\nexample prompt:\n' + (ex.get('example_prompt') or ''))
    for w in ex['windows'][:2]:
        print(f'\nreply for {w["ws"]}-{w["we"]}s:\n{w["reply"]}')
    print('\nexample events:')
    for e in EVS[0]['events'][:5]:
        print(f"  {e['t0']:.1f}-{e['t1']:.1f}s  {e['actor'] or e['actor_text']} | {e['verb']} | "
              f"{e['object'] or e['object_text'] or '-'}  (snapped {e['snapped']})")
left = len(set(MY.key) - on_drive)
print(f'\non Drive: {len(set(MY.key) & on_drive)} of {len(MY)} videos; packs: {len(list(PACKS.glob(TAG + "_*.zip")))}')
RUN.finish('done' if left == 0 else 'partial', notes=f'{left} videos left; gap {GAP}')

{
 "videos_done": 156,
 "videos_in_shard": 156,
 "windows": 1320,
 "parse_rate": 1.0,
 "s_per_window": 3.43,
 "events_per_window": 3.08,
 "events_after_joining": 3459,
 "actor_id_share": 0.977,
 "object_id_share": 0.382,
 "snapped_start_share": 0.136,
 "changes_asked_share": 0.745,
 "confirmed_of_asked": 0.356,
 "star_gap_versions": {
  "derived": 0,
  "pending": 0
 },
 "variant": "main",
 "batch_final": 2,
 "batching_ok": true,
 "updated": "2026-10-17T11:28:47"
}

example prompt:
These 4 frames are from 0.0s to 3.0s of a video, in time order. People and objects are outlined, with their IDs written in the corner: person_1, person_2, person_3, person_38, person_5, person_6, person_7, person_8, person_9, chair_4.
Describe what happens. Reply only in this line format:
CAPTION: <one short sentence, at most 15 words>
EVENT: <who> | <action, a short verb phrase> | <object, or none> | <place, or none>
Write at most 4 EVENT lines, the most important actions first, one action per line. Only rep

## 8. Free the Colab disk (optional)

Only when you are finished with GPU work in this session: set `FREE_DISK_NOW = True` and run this cell. It deletes the downloaded model weights (about 9 GB) and any local video copies. Nothing on Drive is touched, and results are uploaded first. It is off by default so that "Run all" never deletes the model you are using.

In [11]:
FREE_DISK_NOW = False
if FREE_DISK_NOW:
    sync_to_drive(SUBDIRS_03, TAG)
    for n in ['MODEL', 'PROC']:
        globals().pop(n, None)
    free_memory()
    delete_model_cache(VLM_SPEC['id'])
    shutil.rmtree(P['local_videos'] / 'cv', ignore_errors=True)
print('local disk free:', _gb(shutil.disk_usage(LOCAL).free), 'GB')

local disk free: 59.65 GB


## What to send back

The output of section 7 (or `outputs/cv/progress_03_main_shard<N>.json`) after the first session, so the parse rate and speed can be checked before the long runs continue. When both members have finished, notebook 04 builds the memories.